<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week4_Day1_Exercises_XP_Ninja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================
# BLOC 1 — IMPORTATION DES BIBLIOTHÈQUES
# ==============================================================
# On importe uniquement ce dont on a besoin :
#   - load_breast_cancer  : charge le dataset médical intégré à sklearn
#   - train_test_split    : découpe les données en jeu d'entraînement et de test
#   - LogisticRegression  : le modèle de classification qu'on va utiliser
#   - StandardScaler      : normalise les features pour améliorer la convergence
#   - numpy               : calculs numériques (tableaux, opérations vectorielles)
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np


# ==============================================================
# BLOC 2 — DÉFINITION DES FONCTIONS DE MÉTRIQUES (from scratch)
# ==============================================================
# On implémente toutes les métriques manuellement SANS utiliser
# les fonctions prêtes de sklearn (accuracy_score, f1_score, etc.)
# L'objectif pédagogique est de comprendre les formules mathématiques.

def get_confusion_components(y_true, y_pred):
    """
    Extrait les 4 composants fondamentaux de la matrice de confusion.

    Paramètres :
        y_true : liste des vraies étiquettes (valeurs réelles du dataset)
        y_pred : liste des étiquettes prédites par le modèle

    Retourne : TP, TN, FP, FN (entiers)

    Définitions :
        TP (True Positive)  : le modèle prédit 1 ET c'est vraiment 1 → bonne détection
        TN (True Negative)  : le modèle prédit 0 ET c'est vraiment 0 → bonne exclusion
        FP (False Positive) : le modèle prédit 1 MAIS c'est en réalité 0 → fausse alarme
        FN (False Negative) : le modèle prédit 0 MAIS c'est en réalité 1 → cas manqué
    """
    # On parcourt simultanément les vraies valeurs (t) et prédictions (p)
    # et on compte chaque type de résultat avec une compréhension de liste
    TP = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)  # bien prédit positif
    TN = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)  # bien prédit négatif
    FP = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)  # prédit positif à tort
    FN = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)  # prédit négatif à tort
    return TP, TN, FP, FN


def accuracy(y_true, y_pred):
    """
    Calcule l'Exactitude (Accuracy) du modèle.

    Formule : Accuracy = (TP + TN) / (TP + TN + FP + FN)

    Interprétation : proportion de prédictions correctes parmi toutes les prédictions.
    C'est la métrique la plus intuitive, mais elle peut être trompeuse si les
    classes sont déséquilibrées (ex : 95% de bénignes → un modèle qui prédit
    toujours "bénigne" aurait 95% d'accuracy sans rien apprendre !).
    """
    TP, TN, FP, FN = get_confusion_components(y_true, y_pred)
    # Numérateur = toutes les prédictions correctes (positives ET négatives)
    # Dénominateur = total de toutes les prédictions
    return (TP + TN) / (TP + TN + FP + FN)


def precision(y_true, y_pred):
    """
    Calcule la Précision (Precision) du modèle.

    Formule : Precision = TP / (TP + FP)

    Interprétation : parmi tous les cas que le modèle a prédit comme positifs,
    quelle fraction est réellement positive ?
    → Question : "Quand le modèle dit OUI, a-t-il raison ?"
    → Utile quand les fausses alarmes (FP) sont coûteuses (ex : spam détecté à tort).

    Le 'if (TP + FP) > 0' évite une division par zéro si aucun positif n'est prédit.
    """
    TP, TN, FP, FN = get_confusion_components(y_true, y_pred)
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0


def recall(y_true, y_pred):
    """
    Calcule le Rappel (Recall) — aussi appelé Sensibilité ou Taux de Vrais Positifs.

    Formule : Recall = TP / (TP + FN)

    Interprétation : parmi tous les vrais cas positifs (dans le dataset réel),
    quelle fraction le modèle a-t-il réussi à détecter ?
    → Question : "Le modèle rate-t-il des cas positifs ?"
    → Crucial en médecine : un cancer non détecté (FN) est très dangereux.

    Le 'if (TP + FN) > 0' évite une division par zéro si aucun positif réel n'existe.
    """
    TP, TN, FP, FN = get_confusion_components(y_true, y_pred)
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0


def f1_score(y_true, y_pred):
    """
    Calcule le F1-Score : la moyenne harmonique entre Precision et Recall.

    Formule : F1 = 2 × (Precision × Recall) / (Precision + Recall)

    Interprétation : synthèse équilibrée des deux métriques précédentes.
    → Un F1 élevé signifie que le modèle est à la fois précis ET exhaustif.
    → La moyenne harmonique pénalise les extrêmes : si Precision ou Recall est
       très bas, le F1 sera tiré vers le bas — contrairement à la moyenne simple.

    Le 'if (p + r) > 0' évite une division par zéro si les deux sont nuls.
    """
    p = precision(y_true, y_pred)  # calcul de la précision
    r = recall(y_true, y_pred)     # calcul du rappel
    return 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0


# ==============================================================
# BLOC 3 — CHARGEMENT ET PRÉPARATION DES DONNÉES
# ==============================================================

# Chargement du dataset Breast Cancer intégré dans scikit-learn
# Ce dataset contient 569 échantillons et 30 features numériques
# décrivant les caractéristiques de tumeurs (rayon, texture, périmètre, etc.)
data = load_breast_cancer()
X = data.data    # matrice des features : shape (569, 30)
y = data.target  # vecteur des labels   : 1 = maligne (cancer), 0 = bénigne

# --- Normalisation (StandardScaler) ---
# La régression logistique est sensible à l'échelle des variables.
# StandardScaler centre chaque feature à moyenne=0 et écart-type=1.
# Sans normalisation, les features avec de grandes valeurs (ex: périmètre en mm)
# domineraient celles avec de petites valeurs → mauvaise convergence du modèle.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # fit ET transform sur les données complètes

# --- Découpage Train / Test (80% / 20%) ---
# train_test_split divise aléatoirement les données :
#   - X_train, y_train : 80% des données → le modèle apprend dessus
#   - X_test, y_test   : 20% des données → on évalue les prédictions dessus
# random_state=42 garantit la reproductibilité (même résultat à chaque exécution)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,    # 20% réservés pour le test
    random_state=42   # graine aléatoire fixe pour la reproductibilité
)


# ==============================================================
# BLOC 4 — ENTRAÎNEMENT DU MODÈLE ET PRÉDICTIONS
# ==============================================================

# --- Création et entraînement du modèle ---
# LogisticRegression est un modèle de classification binaire.
# Malgré son nom, c'est bien un classifieur (pas une régression au sens strict).
# Il modélise la probabilité d'appartenance à chaque classe via la fonction sigmoïde.
# max_iter=10000 : nombre max d'itérations pour la convergence de l'optimiseur.
model = LogisticRegression(max_iter=10000, random_state=42)

# .fit() entraîne le modèle : il ajuste ses paramètres internes (coefficients)
# pour minimiser l'erreur sur les données d'entraînement.
model.fit(X_train, y_train)

# --- Génération des prédictions ---
# .predict() applique le modèle entraîné aux données de TEST (jamais vues avant).
# Le modèle retourne une étiquette (0 ou 1) pour chaque échantillon du test.
# ⚠ On évalue TOUJOURS sur le jeu de test, jamais sur le jeu d'entraînement,
#   pour mesurer la capacité de généralisation (performances sur données inconnues).
y_pred = model.predict(X_test)


# ==============================================================
# BLOC 5 — CALCUL DES MÉTRIQUES ET AFFICHAGE DES RÉSULTATS
# ==============================================================

# Extraction des composants de la matrice de confusion
# On compare y_test (vérité terrain) avec y_pred (prédictions du modèle)
TP, TN, FP, FN = get_confusion_components(y_test, y_pred)

# Calcul de chaque métrique en appelant nos fonctions définies plus haut
acc  = accuracy(y_test, y_pred)   # exactitude globale
prec = precision(y_test, y_pred)  # précision sur les positifs prédits
rec  = recall(y_test, y_pred)     # rappel sur les vrais positifs
f1   = f1_score(y_test, y_pred)   # score F1 (équilibre précision/rappel)

# --- Affichage structuré des résultats ---
print("=" * 55)
print("   RÉSULTATS - Breast Cancer Wisconsin Dataset")
print("=" * 55)

# Informations générales sur la taille des données
print(f"\n  Dataset  : {X.shape[0]} échantillons, {X.shape[1]} features")
print(f"  Train    : {len(X_train)} échantillons  (80%)")
print(f"  Test     : {len(X_test)} échantillons  (20%)")

# Matrice de confusion : montre la répartition des erreurs et succès
print("\n  --- Matrice de Confusion ---")
print(f"  TP (Vrais Positifs)  = {TP}  ← cancers correctement détectés")
print(f"  TN (Vrais Négatifs)  = {TN}  ← bénignes correctement identifiées")
print(f"  FP (Faux Positifs)   = {FP}   ← bénignes classées à tort comme cancers")
print(f"  FN (Faux Négatifs)   = {FN}   ← cancers NON détectés (les plus dangereux)")

# Métriques calculées avec nos fonctions from scratch
print("\n  --- Métriques d'Évaluation (from scratch) ---")
print(f"  Accuracy   = (TP+TN)/(TP+TN+FP+FN) = {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision  = TP/(TP+FP)             = {prec:.4f}  ({prec*100:.2f}%)")
print(f"  Recall     = TP/(TP+FN)             = {rec:.4f}  ({rec*100:.2f}%)")
print(f"  F1-Score   = 2×(P×R)/(P+R)          = {f1:.4f}  ({f1*100:.2f}%)")

# Interprétation en langage naturel de chaque métrique
print("\n  --- Interprétation des Résultats ---")
print(f"  • Accuracy  : Le modèle prédit correctement {acc*100:.1f}% des cas au total.")
print(f"  • Precision : Quand il prédit un cancer, il a raison {prec*100:.1f}% du temps.")
print(f"  • Recall    : Il détecte {rec*100:.1f}% de TOUS les vrais cancers du jeu de test.")
print(f"  • F1-Score  : Équilibre précision/rappel = {f1:.4f} → excellent (proche de 1.0).")
print(f"\n  ⚠  {FN} cancer(s) manqué(s) (Faux Négatif) → critique en diagnostic médical !")
print("=" * 55)

   RÉSULTATS - Breast Cancer Wisconsin Dataset

  Dataset  : 569 échantillons, 30 features
  Train    : 455 échantillons  (80%)
  Test     : 114 échantillons  (20%)

  --- Matrice de Confusion ---
  TP (Vrais Positifs)  = 70  ← cancers correctement détectés
  TN (Vrais Négatifs)  = 41  ← bénignes correctement identifiées
  FP (Faux Positifs)   = 2   ← bénignes classées à tort comme cancers
  FN (Faux Négatifs)   = 1   ← cancers NON détectés (les plus dangereux)

  --- Métriques d'Évaluation (from scratch) ---
  Accuracy   = (TP+TN)/(TP+TN+FP+FN) = 0.9737  (97.37%)
  Precision  = TP/(TP+FP)             = 0.9722  (97.22%)
  Recall     = TP/(TP+FN)             = 0.9859  (98.59%)
  F1-Score   = 2×(P×R)/(P+R)          = 0.9790  (97.90%)

  --- Interprétation des Résultats ---
  • Accuracy  : Le modèle prédit correctement 97.4% des cas au total.
  • Precision : Quand il prédit un cancer, il a raison 97.2% du temps.
  • Recall    : Il détecte 98.6% de TOUS les vrais cancers du jeu de test.
  